# 👗 Consistent Identity AI — Step 4
## Garment Encoder (Outfit Reference → Texture Features)

**Goal:** Take a photo of any clothing item and extract its texture, color, and
style features so the diffusion model can reproduce it accurately on the person.

**Where we are in the pipeline:**
```
[Ref Image]    → [Identity Encoder ✅] → [Identity Tokens]
[Pose Image]   → [DWPose ✅]           → [Skeleton Map] → [ControlNet]
                                                                  ↓
[Outfit Image] → [Garment Encoder] → [Garment Tokens] → [Diffusion U-Net]
```

**What this notebook covers:**
- ✅ Step 4A: Install dependencies
- ✅ Step 4B: Garment segmentation — isolate clothing from background
- ✅ Step 4C: CLIP-based garment feature extractor
- ✅ Step 4D: Texture-preserving garment encoder (fine-grained features)
- ✅ Step 4E: Garment token projector (maps to diffusion model space)
- ✅ Step 4F: Test with a sample garment image
- ✅ Step 4G: Save garment tokens for Step 5

> ⚡ GPU required: Runtime → Change runtime type → T4 GPU

---
## 📦 Step 4A — Install Dependencies

In [ ]:
# ── Core (skip if continuing from Step 3) ─────────────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate diffusers
!pip install -q opencv-python-headless Pillow matplotlib numpy einops

# ── Segmentation model (removes background from garment photo) ────────────────
!pip install -q rembg[gpu]         # fast background removal via U²-Net
!pip install -q segment-anything   # SAM for precise garment masking

# ── CLIP + feature extraction ─────────────────────────────────────────────────
!pip install -q open_clip_torch    # OpenCLIP — stronger than original CLIP

# ── Scikit for feature analysis ───────────────────────────────────────────────
!pip install -q scikit-learn

print('✅ All dependencies installed.')

---
## 🔧 Step 4B — Imports & GPU Check

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from pathlib import Path
import open_clip
from transformers import AutoImageProcessor, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## ✂️ Step 4C — Garment Segmentation

Before encoding the garment, we isolate the clothing item from its background.
This prevents background colors/textures from polluting the garment embedding.

Two approaches:
- **rembg** (fast, ~100ms) — good for simple product shots
- **SAM** (slower, ~2s) — better for complex backgrounds or worn garments

In [ ]:
from rembg import remove as rembg_remove

class GarmentSegmenter:
    """
    Isolates a clothing item from its background.

    Produces:
        - RGBA image with transparent background
        - Binary mask (white = garment, black = background)
        - Cropped garment on white background (for encoder input)
    """

    def __init__(self, method='rembg'):
        """
        Args:
            method: 'rembg' (fast) or 'color' (no model, heuristic fallback)
        """
        self.method = method
        print(f'✅ GarmentSegmenter ready (method: {method})')

    def segment(self, image: Image.Image) -> dict:
        """
        Args:
            image: PIL Image of a garment (product photo or worn photo)

        Returns dict with:
            'rgba'          : PIL Image RGBA — transparent background
            'mask'          : PIL Image L    — binary garment mask
            'on_white'      : PIL Image RGB  — garment on white background
            'on_black'      : PIL Image RGB  — garment on black background
            'bbox'          : (x1, y1, x2, y2) tight crop of garment
        """
        img_rgb = image.convert('RGB')

        if self.method == 'rembg':
            rgba = rembg_remove(img_rgb)          # PIL RGBA
        else:
            # Fallback: simple center-crop heuristic (no model)
            rgba = self._heuristic_segment(img_rgb)

        # Extract alpha channel as mask
        mask = rgba.split()[-1]                   # PIL Image L (0–255)

        # Compose on white background
        white_bg = Image.new('RGB', rgba.size, (255, 255, 255))
        white_bg.paste(rgba, mask=mask)
        on_white = white_bg

        # Compose on black background
        black_bg = Image.new('RGB', rgba.size, (0, 0, 0))
        black_bg.paste(rgba, mask=mask)
        on_black = black_bg

        # Tight bounding box from mask
        mask_arr = np.array(mask)
        ys, xs = np.where(mask_arr > 128)
        if len(xs) > 0:
            bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
        else:
            bbox = (0, 0, image.width, image.height)

        return {
            'rgba'    : rgba,
            'mask'    : mask,
            'on_white': on_white,
            'on_black': on_black,
            'bbox'    : bbox,
        }

    def _heuristic_segment(self, image: Image.Image) -> Image.Image:
        """
        Simple fallback: uses GrabCut to separate foreground from background.
        Works reasonably well for centered product shots.
        """
        img_arr = np.array(image)
        h, w = img_arr.shape[:2]

        # GrabCut with center rectangle as foreground hint
        mask = np.zeros((h, w), np.uint8)
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        rect = (w//8, h//8, w*6//8, h*6//8)  # center 75% of image

        img_bgr = cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR)
        cv2.grabCut(img_bgr, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)

        # 0=bg, 1=fg, 2=probable_bg, 3=probable_fg
        fg_mask = np.where((mask == 1) | (mask == 3), 255, 0).astype(np.uint8)

        # Smooth the mask edges
        fg_mask = cv2.GaussianBlur(fg_mask, (21, 21), 0)

        rgba = image.convert('RGBA')
        r, g, b, a = rgba.split()
        new_a = Image.fromarray(fg_mask)
        return Image.merge('RGBA', (r, g, b, new_a))


garment_segmenter = GarmentSegmenter(method='rembg')

---
## 🎨 Step 4D — CLIP Garment Feature Extractor

We use **OpenCLIP ViT-H/14** (stronger than the original CLIP) to extract
semantic garment features — what *type* of garment it is, its colors, style.

These are the **coarse features** ("red floral dress", "denim jacket").
Step 4E adds fine-grained texture features on top.

In [ ]:
class CLIPGarmentEncoder:
    """
    Extracts semantic garment features using OpenCLIP ViT-H/14.

    Captures:
        - Garment type (dress, jacket, shirt, trousers...)
        - Dominant colors and patterns
        - Overall style (casual, formal, sporty...)

    Output: 1024-dim L2-normalized embedding
    """

    def __init__(self, model_name='ViT-H-14', pretrained='laion2b_s32b_b79k', device='cuda'):
        self.device = device
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name,
            pretrained=pretrained,
            device=device,
        )
        self.model.eval()
        self.tokenizer = open_clip.get_tokenizer(model_name)
        print(f'✅ CLIPGarmentEncoder loaded ({model_name} / {pretrained})')

    @torch.no_grad()
    def encode_image(self, image: Image.Image) -> np.ndarray:
        """
        Returns 1024-dim image embedding.
        Use the segmented garment (on_white) for cleaner features.
        """
        x = self.preprocess(image).unsqueeze(0).to(self.device)  # (1, 3, 224, 224)
        features = self.model.encode_image(x)                    # (1, 1024)
        features = F.normalize(features, dim=-1)
        return features.squeeze().cpu().numpy()                   # (1024,)

    @torch.no_grad()
    def encode_text(self, description: str) -> np.ndarray:
        """
        Encode a text description of the garment.
        Can be used for text-to-outfit matching.
        """
        tokens = self.tokenizer([description]).to(self.device)    # (1, seq)
        features = self.model.encode_text(tokens)                 # (1, 1024)
        features = F.normalize(features, dim=-1)
        return features.squeeze().cpu().numpy()                   # (1024,)

    def similarity(self, emb_a: np.ndarray, emb_b: np.ndarray) -> float:
        """Cosine similarity between two garment embeddings."""
        return float(np.dot(emb_a, emb_b) /
                     (np.linalg.norm(emb_a) * np.linalg.norm(emb_b) + 1e-8))

    def describe_garment(self, image: Image.Image) -> str:
        """
        Zero-shot garment classification — returns the best matching description
        from a predefined vocabulary.
        """
        candidates = [
            'a red dress', 'a blue dress', 'a white dress', 'a black dress',
            'a floral dress', 'a denim jacket', 'a leather jacket', 'a blazer',
            'a white t-shirt', 'a striped shirt', 'a hoodie', 'a sweater',
            'jeans', 'black trousers', 'shorts', 'a skirt', 'a coat',
            'a suit jacket', 'a crop top', 'a tank top',
        ]

        img_emb = self.encode_image(image)                        # (1024,)
        scores = []
        for desc in candidates:
            txt_emb = self.encode_text(desc)
            scores.append(self.similarity(img_emb, txt_emb))

        best_idx = int(np.argmax(scores))
        return candidates[best_idx], scores[best_idx]


clip_garment_encoder = CLIPGarmentEncoder(device=device)

---
## 🔬 Step 4E — Fine-Grained Texture Encoder (DINOv2 Patches)

CLIP gives us *semantics* (what it is). DINOv2 patch tokens give us *texture*
(exactly how it looks — weave pattern, fabric grain, print details).

We extract **all patch tokens** (not just the CLS token) to preserve spatial
texture information. This is what makes "the exact same fabric" transfer correctly.

In [ ]:
class TextureEncoder:
    """
    Extracts fine-grained texture and pattern features from a garment image.

    Uses DINOv2 ViT-B/14 patch tokens (not just CLS):
        - Input: 224×224 image → 16×16 patches
        - Output: 256 patch tokens × 768 dims = spatial texture map
        - We also return the mean-pooled version as a compact descriptor

    This captures fabric weave, print patterns, and fine color gradients
    that CLIP's CLS token misses.
    """

    def __init__(self, model_name='facebook/dinov2-base', device='cuda'):
        self.device = device
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        print(f'✅ TextureEncoder loaded ({model_name})')

    @torch.no_grad()
    def encode(self, image: Image.Image) -> dict:
        """
        Args:
            image: PIL Image of garment (on white background preferred)

        Returns dict with:
            'patch_tokens'  : np.ndarray (256, 768) — spatial texture map
            'cls_token'     : np.ndarray (768,)     — global texture descriptor
            'mean_pooled'   : np.ndarray (768,)     — mean of patch tokens
        """
        inputs = self.processor(images=image, return_tensors='pt').to(self.device)
        outputs = self.model(**inputs, output_hidden_states=True)

        last_hidden = outputs.last_hidden_state  # (1, 257, 768)  [CLS + 256 patches]

        cls_token    = last_hidden[:, 0,  :]     # (1, 768)
        patch_tokens = last_hidden[:, 1:, :]     # (1, 256, 768)

        # L2 normalize
        cls_token    = F.normalize(cls_token,    dim=-1)
        patch_tokens = F.normalize(patch_tokens, dim=-1)
        mean_pooled  = F.normalize(patch_tokens.mean(dim=1), dim=-1)

        return {
            'patch_tokens' : patch_tokens.squeeze().cpu().numpy(),   # (256, 768)
            'cls_token'    : cls_token.squeeze().cpu().numpy(),      # (768,)
            'mean_pooled'  : mean_pooled.squeeze().cpu().numpy(),    # (768,)
        }

    def patch_similarity_map(self, texture_a: dict, texture_b: dict) -> np.ndarray:
        """
        Computes patch-level similarity between two garments.
        Returns a 16×16 heatmap showing which regions match.
        Useful for debugging texture transfer accuracy.
        """
        pa = texture_a['patch_tokens']   # (256, 768)
        pb = texture_b['patch_tokens']   # (256, 768)

        # Dot product similarity per patch pair
        sim = (pa * pb).sum(axis=-1)     # (256,) cosine sim per patch
        return sim.reshape(16, 16)       # 16×16 spatial map


texture_encoder = TextureEncoder(device=device)

---
## 🔗 Step 4F — Garment Token Projector

The diffusion model's cross-attention expects tokens of shape `(N, 768)` in a specific
embedding space. This projector maps our garment features into that space so the
U-Net can attend to them during generation.

In [ ]:
class GarmentTokenProjector(nn.Module):
    """
    Projects garment features into the diffusion model's cross-attention space.

    Input  : CLIP features (1024,) + Texture features (768,)
    Output : N garment tokens of shape (N, 768)
             where N = num_tokens (default 16 — one per major garment region)

    These tokens are appended to the text prompt tokens in the U-Net's
    cross-attention layers, so the model learns to attend to garment texture
    while also following the text prompt.

    Architecture:
        clip   (1024) ─┐
                        ├─ concat (1792) ─ LayerNorm ─ MLP ─ reshape → (N, 768)
        texture (768) ─┘
    """

    def __init__(
        self,
        clip_dim    : int = 1024,
        texture_dim : int = 768,
        num_tokens  : int = 16,
        token_dim   : int = 768,   # must match SDXL cross-attention dim
    ):
        super().__init__()
        self.num_tokens = num_tokens
        self.token_dim  = token_dim
        fused_dim = clip_dim + texture_dim  # 1792

        self.clip_norm    = nn.LayerNorm(clip_dim)
        self.texture_norm = nn.LayerNorm(texture_dim)

        self.mlp = nn.Sequential(
            nn.Linear(fused_dim, fused_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(fused_dim, num_tokens * token_dim),
        )
        self.output_norm = nn.LayerNorm(token_dim)

    def forward(
        self,
        clip_feat   : torch.Tensor,   # (B, 1024) or (1024,)
        texture_feat: torch.Tensor,   # (B, 768)  or (768,)
    ) -> torch.Tensor:                # (B, N, 768)

        if clip_feat.dim() == 1:
            clip_feat    = clip_feat.unsqueeze(0)
        if texture_feat.dim() == 1:
            texture_feat = texture_feat.unsqueeze(0)

        B = clip_feat.shape[0]

        clip_feat    = self.clip_norm(clip_feat)       # (B, 1024)
        texture_feat = self.texture_norm(texture_feat) # (B, 768)

        fused = torch.cat([clip_feat, texture_feat], dim=-1)  # (B, 1792)

        tokens = self.mlp(fused)                       # (B, N * 768)
        tokens = tokens.view(B, self.num_tokens, self.token_dim)  # (B, N, 768)
        tokens = self.output_norm(tokens)              # (B, N, 768)

        return tokens


garment_projector = GarmentTokenProjector(
    clip_dim=1024,
    texture_dim=768,
    num_tokens=16,
    token_dim=768,
).to(device)
garment_projector.eval()

total_params = sum(p.numel() for p in garment_projector.parameters())
print(f'✅ GarmentTokenProjector ready')
print(f'   Parameters : {total_params:,}')
print(f'   Output     : 16 tokens × 768 dims')

---
## 🎯 Step 4G — Full Garment Encoder Pipeline

One class that wraps segmentation → CLIP → texture → projection.

In [ ]:
class GarmentEncoder:
    """
    Full garment encoding pipeline.

    Input  : PIL Image of a clothing item
    Output : 16 garment tokens of shape (16, 768) ready for cross-attention

    Internally:
        1. Segment    — remove background via rembg
        2. CLIP       — semantic features (1024-dim)
        3. DINOv2     — texture/patch features (768-dim)
        4. Projector  — map to 16 × 768 garment tokens
    """

    def __init__(self, segmenter, clip_enc, texture_enc, projector, device='cuda'):
        self.segmenter   = segmenter
        self.clip_enc    = clip_enc
        self.texture_enc = texture_enc
        self.projector   = projector
        self.device      = device

    def encode(self, image: Image.Image) -> dict:
        """
        Returns dict with:
            'garment_tokens'  : np.ndarray (16, 768) — ready for cross-attention
            'clip_features'   : np.ndarray (1024,)   — semantic features
            'texture_features': np.ndarray (768,)    — texture features
            'segmented'       : PIL Image            — garment on white bg
            'description'     : str                  — auto-detected garment type
            'clip_score'      : float                — confidence of description
        """
        # ── 1. Segment garment ────────────────────────────────────────────────
        seg = self.segmenter.segment(image)
        garment_img = seg['on_white']  # clean garment on white bg

        # ── 2. CLIP features ──────────────────────────────────────────────────
        clip_feat = self.clip_enc.encode_image(garment_img)   # (1024,)
        description, score = self.clip_enc.describe_garment(garment_img)

        # ── 3. Texture features ───────────────────────────────────────────────
        texture_result = self.texture_enc.encode(garment_img)
        texture_feat   = texture_result['mean_pooled']         # (768,)

        # ── 4. Project to garment tokens ──────────────────────────────────────
        clip_t    = torch.tensor(clip_feat,    dtype=torch.float32).to(self.device)
        texture_t = torch.tensor(texture_feat, dtype=torch.float32).to(self.device)

        with torch.no_grad():
            tokens = self.projector(clip_t, texture_t)  # (1, 16, 768)

        tokens_np = tokens.squeeze(0).cpu().numpy()     # (16, 768)

        return {
            'garment_tokens'  : tokens_np,
            'clip_features'   : clip_feat,
            'texture_features': texture_feat,
            'segmented'       : garment_img,
            'description'     : description,
            'clip_score'      : score,
        }

    def save(self, result: dict, path: str):
        """Save garment tokens and features to .npz"""
        np.savez(
            path,
            garment_tokens   = result['garment_tokens'],
            clip_features    = result['clip_features'],
            texture_features = result['texture_features'],
        )
        result['segmented'].save(path.replace('.npz', '_segmented.png'))
        print(f'💾 Garment saved → {path}.npz')

    @staticmethod
    def load(path: str) -> dict:
        data = np.load(path)
        return {
            'garment_tokens'  : data['garment_tokens'],
            'clip_features'   : data['clip_features'],
            'texture_features': data['texture_features'],
        }


garment_encoder = GarmentEncoder(
    segmenter   = garment_segmenter,
    clip_enc    = clip_garment_encoder,
    texture_enc = texture_encoder,
    projector   = garment_projector,
    device      = device,
)
print('✅ Full GarmentEncoder pipeline assembled!')

---
## 🧪 Step 4H — Test with Sample Garment Image

In [ ]:
# ── Option A: Upload your own garment photo ────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# garment_path = list(uploaded.keys())[0]
# garment_image = Image.open(garment_path).convert('RGB')

# ── Option B: Synthetic placeholder garment ────────────────────────────────────
# Creates a simple colorful shirt shape for testing
def make_test_garment(size=512):
    img = Image.new('RGB', (size, size), (245, 245, 245))
    d = ImageDraw.Draw(img)
    cx, cy = size // 2, size // 2

    # T-shirt silhouette
    body = [cx-100, cy-80, cx+100, cy-80, cx+120, cy-40,
            cx+80, cy-20, cx+80, cy+120, cx-80, cy+120, cx-80, cy-20, cx-120, cy-40]
    d.polygon(body, fill=(220, 60, 60), outline=(180, 40, 40))

    # Collar
    d.ellipse([cx-30, cy-100, cx+30, cy-60], fill=(180, 40, 40))

    # Sleeves
    d.polygon([cx-80, cy-20, cx-80, cy-60, cx-160, cy-40, cx-140, cy-10], fill=(220, 60, 60))
    d.polygon([cx+80, cy-20, cx+80, cy-60, cx+160, cy-40, cx+140, cy-10], fill=(220, 60, 60))

    # Stripes pattern
    for y in range(cy-20, cy+120, 20):
        d.line([cx-75, y, cx+75, y], fill=(200, 50, 50), width=2)

    return img

garment_image = make_test_garment(512)
print('ℹ️  Using synthetic garment. Uncomment Option A to use a real garment photo.')
print(f'   Image size: {garment_image.size}')

In [ ]:
# ── Run the full garment encoder ───────────────────────────────────────────────
garment_result = garment_encoder.encode(garment_image)

print('\n── Garment Encoding Results ────────────────────────────────')
print(f'Auto description  : "{garment_result["description"]}" (score: {garment_result["clip_score"]:.3f})')
print(f'CLIP features     : {garment_result["clip_features"].shape}')    # (1024,)
print(f'Texture features  : {garment_result["texture_features"].shape}') # (768,)
print(f'Garment tokens    : {garment_result["garment_tokens"].shape}')   # (16, 768)

In [ ]:
# ── Visualize ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Garment Encoding Visualization', fontsize=14, fontweight='bold')

# Original garment
axes[0].imshow(garment_image)
axes[0].set_title('Input garment')
axes[0].axis('off')

# Segmented garment
axes[1].imshow(garment_result['segmented'])
axes[1].set_title(f'Segmented\n"{garment_result["description"]}"')
axes[1].axis('off')

# Garment token heatmap (16 tokens × 768 dims)
tokens_vis = garment_result['garment_tokens']            # (16, 768)
tokens_norm = (tokens_vis - tokens_vis.mean()) / (tokens_vis.std() + 1e-6)
im = axes[2].imshow(tokens_norm, cmap='coolwarm', aspect='auto', vmin=-2, vmax=2)
axes[2].set_title('Garment tokens\n(16 × 768)')
axes[2].set_xlabel('Token dimension (768)')
axes[2].set_ylabel('Token index (16)')
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.savefig('garment_encoding.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Test garment similarity between two items ──────────────────────────────────
# In real use: similar garments should score > 0.7, different ones < 0.4

garment_b = make_test_garment(512)   # same garment → should be ~1.0
result_b   = garment_encoder.encode(garment_b)

sim = clip_garment_encoder.similarity(
    garment_result['clip_features'],
    result_b['clip_features']
)
print(f'Garment similarity (same image, two runs): {sim:.4f}')

---
## 💾 Step 4I — Save for Step 5

In [ ]:
import json

# Save garment tokens
garment_encoder.save(garment_result, 'my_garment')

# Update pipeline config
try:
    with open('pipeline_config.json') as f:
        config = json.load(f)
except FileNotFoundError:
    config = {}

config.update({
    'garment_file'       : 'my_garment.npz',
    'garment_description': garment_result['description'],
    'garment_num_tokens' : 16,
    'garment_token_dim'  : 768,
})
with open('pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Step 4 complete!')
print()
print('   Saved files:')
print('     my_garment.npz              ← garment tokens + features')
print('     my_garment_segmented.png    ← clean garment on white bg')
print('     pipeline_config.json        ← updated config')
print()
print('   Next → Step 5: Fuse identity + pose + garment into one inference call!')

---
## 📋 Summary

| Component | Model | Output | What it captures |
|-----------|-------|--------|------------------|
| `GarmentSegmenter` | rembg (U²-Net) | RGBA mask | Clean garment silhouette |
| `CLIPGarmentEncoder` | OpenCLIP ViT-H/14 | 1024-dim | Garment type, color, style |
| `TextureEncoder` | DINOv2 ViT-B/14 | 256×768 patches | Fabric weave, print, texture |
| `GarmentTokenProjector` | 2-layer MLP | 16×768 tokens | Diffusion-ready cross-attention tokens |
| `GarmentEncoder` | Full pipeline | 16×768 | One-stop encode + save |

## 🗺️ Full pipeline status

```
Step 1+2  ✅  Identity encoder  →  my_identity.npz   (512-dim face+body)
Step 3    ✅  Pose control      →  my_pose.png        (skeleton map)
Step 4    ✅  Garment encoder   →  my_garment.npz     (16 × 768 tokens)
Step 5    ⬅️  Fusion pipeline   →  inject all 3 into SDXL → generate!
Step 6        Post-processing   →  face restore + consistency check
Step 7        Gradio UI         →  interactive web app
```

> 💡 The projector weights are randomly initialized — they'll be fine-tuned
> in Step 5 alongside the IP-Adapter identity injection using a combined loss.